<a href="https://colab.research.google.com/github/AmplMrrr/compling-HW/blob/main/%D0%9F%D0%BE%D1%81%D1%82%D1%80%D0%BE%D0%B5%D0%BD%D0%B8%D0%B5_RAG_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D1%8B_%D1%81_%D0%B8%D1%81%D0%BF%D0%BE%D0%BB%D1%8C%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5%D0%BC_LangChain_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Тема:** RAG (Retrieval-Augmented Generation) с фреймворком LangChain


Разработка RAG-пайплайна


**Проект выполнили:** Амплеева Марианна, Белоусова-Гельцер Кристина, Федорчук Дмитрий

Код загрузили один раз (в репозиторий Марианны)

**Задачи:**
* Загрузить набор текстовых документов (например, статей из датасета arXiv Dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv)
* Разбить текст на чанки с помощью Langchain text splitter
* Создать векторный индекс с помощью FAISS и sentence-transformers
* Реализовать langchain-цепочку, которая производим семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)
* Протестировать систему на нескольких вопросах, оценить качество ответов


**Библиотеки:** langchain, huggingface, faiss-cpu, sentence-transformers

**Ожидаемый результат:** Colab-ноутбук с рабочим прототипом наукоёмкой (например, разработанной на основе текстов ArXiv) RAG-системы, примерами её ответов и качественным анализом, представленным в текстовых блоках


## Загрузить набор текстовых документов

### Датасет с метаданными к статьям

In [ ]:
# Импортируем KaggleHub для загрузки датасетов и pandas для работы с данными
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "arxiv-metadata-oai-snapshot.json" # Указываем локальный файл с  метаданными датасета arXiv из Kaggle

# Загружаем первые 1000 строк датасета
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "Cornell-University/arxiv",
    file_path,
    pandas_kwargs={"lines": True, "nrows": 1000},
)

# Создаём сырой id для ссылок
df["id_raw"] = df["id"].astype(str)

print("Колонки:", df.columns.tolist())
print(df[["id", "id_raw"]].head())

/tmp/ipykernel_454/1052440118.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 4.80G/4.80G [00:45<00:00, 114MB/s]


Колонки: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed', 'id_raw']
         id    id_raw
0  704.0001  704.0001
1  704.0002  704.0002
2  704.0003  704.0003
3  704.0004  704.0004
4  704.0005  704.0005


In [ ]:
# Смотрим, что получилось
df.head(2)

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed,id_raw
0,704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,...",704.0001
1,704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]",704.0002


### Подбор статей

внимание на столбец id: по нему можно добраться до самой статьи

 https://arxiv.org/abs/{id}: посмотреть страницу статьи с абстрактом

 https://arxiv.org/pdf/{id}: скачать

In [ ]:
# Пример без фильтра (просто чтобы проверить механизм ссылок)
df_filtered = df.copy()


ids = df_filtered["id_raw"].tolist()
print(f"Найдено {len(ids)} id")

if len(ids) == 0:
    print("После фильтрации не осталось ни одной строки")
else:
    abs_urls = [f"https://arxiv.org/abs/{i}" for i in ids]
    pdf_urls = [f"https://arxiv.org/pdf/{i}.pdf" for i in ids]

    print("Пример ссылки на страницу:", abs_urls[0])
    print("Пример ссылки на PDF:", pdf_urls[0])

Найдено 1000 id
Пример ссылки на страницу: https://arxiv.org/abs/704.0001
Пример ссылки на PDF: https://arxiv.org/pdf/704.0001.pdf


### Загрузка

In [ ]:
!pip install langchain_community langchain_text_splitters pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
# Подготавливаем инструменты LangChain для загрузки PDF, arXiv и разбиения текста на чанки
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import ArxivLoader
!pip install arxiv

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 8.3 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=384b63542c354f1aa3cef2e190affaa3b230a62c5e804ae7925d550450942dcd
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
# Загружаем документы
base_url = "https://arxiv.org/pdf"

documents = []

print("Количество id:", len(ids))
print("Первые id:", ids[:1000])


print("\n  ArxivLoader ")
try:
    arxiv_loader = ArxivLoader(
        query="cs.CL",
        load_max_docs=1000,
        load_all_available_meta=True
    )
    arxiv_docs = arxiv_loader.load()
    documents.extend(arxiv_docs)
    print(f"Загружено {len(arxiv_docs)} документов через ArxivLoader")
except Exception as e:
    print(f"ArxivLoader не сработал: {e}")

# Этот цикл нужен  если arxivloader не дал документов, и мы пробуем абстракты из df_filtered
if len(documents) == 0:
    print("Используем абстракты из метаданных")
    from langchain_core.documents import Document

    for idx, row in df_filtered.head(100).iterrows():
        title = row.get('titles', row.get('title', 'No title'))
        abstract = row.get('summaries', row.get('abstract', 'No abstract'))
        doc_id = str(row['id'])

        text = f"Title: {title}\n\nAbstract: {abstract}"

        doc = Document(
            page_content=text,
            metadata={
                "id": doc_id,
                "title": title,
                "source": "arXiv metadata"
            }
        )
        documents.append(doc)

    print(f"Создано {len(documents)} документов из метаданных")

# Смотрим, получилось ли загрузить документы
print(f"Итог: {len(documents)} страниц (Document-объектов)")

if len(documents) == 0:
    print("Не удалось загрузить документы. Проверь df_filtered.")
else:
    print("Пример документа:")
    print("Metadata:", documents[0].metadata)
    print("\nContent preview:")
    print(documents[0].page_content[:500])

Количество id: 1000
Первые id: ['704.0001', '704.0002', '704.0003', '704.0004', '704.0005', '704.0006', '704.0007', '704.0008', '704.0009', '704.001', '704.0011', '704.0012', '704.0013', '704.0014', '704.0015', '704.0016', '704.0017', '704.0018', '704.0019', '704.002', '704.0021', '704.0022', '704.0023', '704.0024', '704.0025', '704.0026', '704.0027', '704.0028', '704.0029', '704.003', '704.0031', '704.0032', '704.0033', '704.0034', '704.0035', '704.0036', '704.0037', '704.0038', '704.0039', '704.004', '704.0041', '704.0042', '704.0043', '704.0044', '704.0045', '704.0046', '704.0047', '704.0048', '704.0049', '704.005', '704.0051', '704.0052', '704.0053', '704.0054', '704.0055', '704.0056', '704.0057', '704.0058', '704.0059', '704.006', '704.0061', '704.0062', '704.0063', '704.0064', '704.0065', '704.0066', '704.0067', '704.0068', '704.0069', '704.007', '704.0071', '704.0072', '704.0073', '704.0074', '704.0075', '704.0076', '704.0077', '704.0078', '704.0079', '704.008', '704.0081', '704

## Разбить на чанки

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Настройка под абстракты научных статей
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,           # размер чанка в символах
    chunk_overlap=100,        # перекрытие
    length_function=len,      # длина
    add_start_index=True      # индекс начала в метаданных
)

# Разбиваем на чанки documentes через split_documents
chunks = text_splitter.split_documents(documents)

print(f"Создано {len(chunks)} чанков из {len(documents)} документов")
print(f"Средний размер чанка: {sum(len(c.page_content) for c in chunks) // len(chunks):.0f} символов")

# Пример чанка с метаданными
print("\n пример чанка")
print("Metadata:", chunks[0].metadata)
print("\nContent:")
print(chunks[0].page_content[:400] + "...")

# Статистика по размерам чанков
chunk_sizes = [len(c.page_content) for c in chunks]
print(f"\nСтатистика чанков:")
print(f"  Min: {min(chunk_sizes)} символов")
print(f"  Max: {max(chunk_sizes)} символов")
print(f"  95-й процентиль: {sorted(chunk_sizes)[int(0.95*len(chunk_sizes))]} символов")

Создано 210 чанков из 100 документов
Средний размер чанка: 444 символов

 пример чанка
Metadata: {'id': '704.0001', 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'source': 'arXiv metadata', 'start_index': 0}

Content:
Title: Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies...

Статистика чанков:
  Min: 33 символов
  Max: 798 символов
  95-й процентиль: 784 символов


## Создать векторный индекс с помощью faiss-cpu и sentence-transformers

In [ ]:
!pip install faiss-cpu sentence-transformers langchain-huggingface -q #без FAISS все галлюцинирует, и rag не состоится, подключаем

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.4 MB/s eta 0:00:00


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Создаём эмбеддинги
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
)

# Создаём векторное хранилище
print("Создаём векторное хранилище FAISS...")
vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Обработано чанков: {vectorstore.index.ntotal}")
print(f"Создан FAISS индекс с {vectorstore.index.ntotal} векторами")

# Проверяем, что всё работает)
test_query = "neural networks"
relevant_docs = vectorstore.similarity_search(test_query, k=3)
print(f"Тест поиска: '{test_query}'")
for i, doc in enumerate(relevant_docs):
    print(f"{i+1}. {doc.metadata.get('title', 'No title')[:60]}...")
    print(f"   {doc.page_content[:150]}...\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Создаём векторное хранилище FAISS...
Обработано чанков: 210
Создан FAISS индекс с 210 векторами

=== тест поиска: 'neural networks' ===
1. Intelligent location of simultaneously active acoustic emiss...
   Abstract:   The intelligent acoustic emission locator is described in Part I, while Part
II discusses blind source separation, time delay estimation a...

2. A general approach to statistical modeling of physical laws:...
   Title: A general approach to statistical modeling of physical laws:
  nonparametric regression...

3. A general approach to statistical modeling of physical laws:...
   Abstract:   Statistical modeling of experimental physical laws is based on the
probability density function of measured variables. It is expressed by
...



## Реализовать цепочку

In [ ]:
!pip install langchain_core langchain_classic -q

In [ ]:

from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate

In [ ]:
!pip install langchain_core langchain_classic langchain_huggingface faiss-cpu sentence-transformers transformers torch accelerate -q

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import json
from huggingface_hub import login # НЕ РАБОТАЕТ, ЦУКЕРБЕРГ ЖАДИНА


def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(str(text)) > threshold else str(text)

# Сначала у нас была модель distillgpt2, но она оказалась нерабоспособной. Мы попробовали ламу, но эта модель выдаёт ошибку "gated repository". Попробовали сделать токен на HF, но не вышло. Берем Квен.
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 4 # Топ-К отвечает за контекст, а именно за чанки документов. Если будет больше, то увеличивается шум и длительность модели, а у нас она итак по 5-7 минут грузит.
QUESTION = "Какие оптимизаторы используются в машинном обучении?"

PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Ты ассистент. Отвечай только на основе предоставленного контекста.
<|eot_id|><|start_header_id|>user<|end_header_id|> #это не моя прихоть, это требование Qwen

Контекст:
{context}

Вопрос: {input}
<|eot_id|><|start_header_id|>assistant<|end_header_id|> #это тоже

Ответ:"""

# Документы
SAMPLE_DOCS = [
    Document(page_content="Методы обучения: SGD, Adam, RMSprop. Градиентный спуск.", metadata={"source": "arxiv_001"}),
    Document(page_content="Adam optimizer для компьютерного зрения.", metadata={"source": "arxiv_002"}),
]

# Vectorstore
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID)
vectorstore = FAISS.from_documents(SAMPLE_DOCS, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# Пайплайн

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150, #Qwen в этом плане хорош, потому что первая модель даже ничего не могла сказать в тестах.
    temperature=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
    device=0 if torch.cuda.is_available() else -1
)

llm = HuggingFacePipeline(pipeline=pipe)

# RAG цепочка
question_answer_chain = create_stuff_documents_chain(
    llm,
    PromptTemplate.from_template(PROMPT)
)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Вывод
resp_dict = rag_chain.invoke({"input": QUESTION})
clipped_answer = clip_text(resp_dict["answer"], threshold=350)
print(f"вопрос:\n{resp_dict['input']}\n\n ответ:\n{clipped_answer}")
print(f"\n источники ({len(resp_dict['context'])}):")
for i, doc in enumerate(resp_dict["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
    for key in doc.metadata:
        if key != "pk":
            val = doc.metadata.get(key)
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


вопрос:
Какие оптимизаторы используются в машинном обучении?

 ответ:
 В машинном обучении часто используются следующие оптимизаторы:

1. **Градиентный спуск (SGD)** - простой и эффективный метод, который просто минимизирует функцию потерь с помощью градиента.

2. **Адам (Adam)** - обновляющийся оптимизатор, который учитывает информацию о среднем значении градиента и среднем значении квадратов градиентов.

3. **РМСПР...

 источники (2):

Source 1:
  text: "Adam optimizer \u0434\u043b\u044f \u043a\u043e\u043c\u043f\u044c\u044e\u0442\u0435\u0440\u043d\u043e\u0433\u043e \u0437\u0440\u0435\u043d\u0438\u044f."
  source: arxiv_002

Source 2:
  text: "\u041c\u0435\u0442\u043e\u0434\u044b \u043e\u0431\u0443\u0447\u0435\u043d\u0438\u044f: SGD, Adam, RMSprop. \u0413\u0440\u0430\u0434\u0438\u0435\u043d\u0442\u043d\u044b\u0439 \u0441\u043f\u0443\u0441\u043a."
  source: arxiv_001


## Протестировать на нескольких примерах, оченить качество

In [ ]:
test_questions = [
    "Какие оптимизаторы используются в машинном обучении?",  # Фактический
    "Какие преимущества нейронных сетей?",                  # Обобщающий
    "Что такое Adam optimizer в контексте?"                 # Уточняющий
]


results = []
for question in test_questions:
    response = rag_chain.invoke({"input": question})

    print(f"\n{'='*60}")
    print(f"ВОПРОС: {question}")
    print(f"ОТВЕТ: {response['answer'][:400]}...")
    print(f"КОНТЕКСТ ({len(response['context'])} чанка):")

    for i, doc in enumerate(response['context']):
        print(f"  {i+1}. {doc.metadata.get('title', 'No title')[:50]}...")
        print(f"     {doc.page_content[:200]}...")

    # Простая эвристика качества
    relevance_score = sum('оптимизатор' in doc.page_content.lower() or 'нейрон' in doc.page_content.lower() for doc in response['context']) / len(response['context'])
    print(f"Эвристика релевантности: {relevance_score:.2f}")

    results.append({
        'question': question,
        'answer': response['answer'],
        'context_len': len(response['context']),
        'relevance': relevance_score
    })

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ВОПРОС: Какие оптимизаторы используются в машинном обучении?
ОТВЕТ:  В машинном обучении часто используются следующие оптимизаторы:

1. **Градиентный спуск (SGD)** - простой и эффективный метод, который просто минимизирует функцию потерь с помощью градиента.

2. **Адам (Adam)** - обновляющийся оптимизатор, который учитывает информацию о среднем значении градиента и среднем значении квадратов градиентов.

3. **РМСПРОП (RMSProp)** - аналогичен Адаму, но использует с...
КОНТЕКСТ (2 чанка):
  1. No title...
     Adam optimizer для компьютерного зрения....
  2. No title...
     Методы обучения: SGD, Adam, RMSprop. Градиентный спуск....
Эвристика релевантности: 0.00


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ВОПРОС: Какие преимущества нейронных сетей?
ОТВЕТ:  Нейронные сети обладают рядом важных преимуществ:

1. Пространственное восприятие: Они способны анализировать и интерпретировать сложные пространственные данные, что делает их идеальными для задач, связанных с изображениями и видео.

2. Обучение на больших данных: Нейронные сети могут обучаться на огромных объемах данных, что позволяет им улучшать свои результаты в условиях ограниченной ресурсами...
КОНТЕКСТ (2 чанка):
  1. No title...
     Методы обучения: SGD, Adam, RMSprop. Градиентный спуск....
  2. No title...
     Adam optimizer для компьютерного зрения....
Эвристика релевантности: 0.00

ВОПРОС: Что такое Adam optimizer в контексте?
ОТВЕТ:  Adam optimizer - это метод оптимизации, используемый в контексте компьютерного зрения и других областях машинного обучения. Он представляет собой модификацию градиентного спуска, которая учитывает информацию о среднем квадратичном отклонении (MSE) и среднем квадратичном отклонении по времени